# Research Question 1: 

The goal: for each ecoregion, estimate how strongly and in which direction pre-season climate anomaly predicts fire season onset timing (β, days/°C), using statsmodels.MixedLM with partial pooling across ecoregions.


## 0. Setup
### 0.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from scipy.stats import spearmanr
from scipy.stats import t as t_dist
import json
from shapely.geometry import shape
import geopandas as gpd

import os
import warnings
import logging
print('Libraries loaded.')

pd.set_option('display.expand_frame_repr', False)

### 0.2 Run Configuration

In [ ]:
# !!!IMPORTANT!!! 
# Folders and some file names are dependent on this

RUN_LABEL   = 'med_basin'  # short name for this run
RUN_VERSION = 'v4'         # increment this for each new run
RUN_NAME   = f'{RUN_LABEL}_{RUN_VERSION}'

### 0.3 Paths

In [ ]:
RESPONSE_VARS  = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
LAG_WINDOWS    = [30, 45, 60, 90]
PREDICTOR_SETS = {
    'temp':      ['temp_anomaly_{lag}d'],
    'precip':    ['precip_anomaly_{lag}d'],
    'both':      ['temp_anomaly_{lag}d', 'precip_anomaly_{lag}d'],
}
GROUP_COL = 'eco_id'

# File paths

BASE_OUT_DIR = 'C:/Users/ibekar/Documents/GitProjects/TGPF' # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

input_dir_processed = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'inputs')
results_dir = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'outputs', 'RQ1')
plot_dir = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'outputs', 'RQ1', 'plots')
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)

print(f'Base output directory: {BASE_OUT_DIR}')
print(f'Input directory: {input_dir_processed}')
print(f'Results directory: {results_dir}')
print(f'Plot directory: {plot_dir}')



## 1. Data Loading

In [ ]:
# Load geometries

geo_path = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'eco_geometries.json')

with open(geo_path) as f:
    data = json.load(f)

gdf = gpd.GeoDataFrame(
    [{'eco_id': d['eco_id'], 'eco_name': d['eco_name']} for d in data],
    geometry=[shape(d['geometry']) for d in data],
    crs='EPSG:4326'
)
print(f'Loaded {len(gdf)} ecoregion geometries.')

In [ ]:
# Load files
df = pd.read_csv(os.path.join(input_dir_processed, 'era5_anomalies_by_lag.csv'))

print(df)

In [ ]:
print(df[df.isnull().any(axis=1)])

# NA Check 
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("There is no NA")
else:
    print(null_counts)

In [ ]:
# Drop NA
df = df.dropna(subset=[col for col in df.columns if 'anomaly' in col])

# Sanity check 
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if null_counts.empty:
    print("There is no NA")
else:
    print(null_counts)

## 2. Model Fitting
### 2.1 MixedLM Function

In [ ]:
# Model fitting function

def fit_mixedlm(df, response, predictors, group_col):
    df_subset = df[[response] + predictors + [group_col]].dropna()

    # Standardize predictors
    scaler_parameters = {}
    for predictor in predictors:
        mean = df_subset[predictor].mean()
        std = df_subset[predictor].std()
        df_subset[predictor + '_z'] = (df_subset[predictor] - mean) / std
        scaler_parameters[predictor] = {
            'mean': mean,
            'std': std
        }

    z_predictors = [pred + '_z' for pred in predictors]
    formula = response + ' ~ ' + ' + '.join(z_predictors)
    re_formula = ' ~ ' + ' + '.join(z_predictors)
    fallback = False

    try:
        model = MixedLM.from_formula(formula, groups=df_subset[group_col], re_formula=re_formula, data=df_subset)
        results = model.fit(reml=False)
        convergence_ok = results.converged
    except Exception as e:
        print(f"Model fitting failed for {response} with predictors {predictors}. Attempting fallback to random intercept only.")
        fallback = True
        model = MixedLM.from_formula(formula, groups=df_subset[group_col], data=df_subset)
        results = model.fit()
        convergence_ok = results.converged

    # Extract fixed effects
    ci = results.conf_int()
    fixed_effects = {}
    for zp in z_predictors:
        original_name = zp[:-2]  # Remove '_z' suffix
        coef = results.params[zp]
        se = results.bse[zp]
        pval = results.pvalues[zp]
        ci_low = ci.loc[zp, 0]
        ci_hi = ci.loc[zp, 1]

        fixed_effects[original_name] = {
            'coef' : coef,
            'se'   : se,
            'pval' : pval,
            'ci_low': ci_low,
            'ci_hi' : ci_hi,
        }

    # get eco_id and re_dict for random effects
    random_effects = []
    for group, re_params in results.random_effects.items():
        row = {'eco_id': group}
        for zp in z_predictors:
            total_slope = fixed_effects[zp[:-2]]['coef'] + re_params.get(zp, 0)
            row[zp[:-2] + '_slope'] = total_slope
        random_effects.append(row)

    # Extract model fit metrics
    aic = results.aic
    log_likelihood = results.llf

    # Back-transform slopes to original units
    for fe in fixed_effects:
        std = scaler_parameters[fe]['std']
        fixed_effects[fe]['coef_unscaled'] = fixed_effects[fe]['coef'] / std
        fixed_effects[fe]['ci_low_unscaled'] = fixed_effects[fe]['ci_low'] / std
        fixed_effects[fe]['ci_hi_unscaled'] = fixed_effects[fe]['ci_hi'] / std

    for row in random_effects:
        for pred in predictors:
            std = scaler_parameters[pred]['std']
            row[pred + '_slope_original'] = row[pred + '_slope'] / std

    return {
        'fixed_effects':  fixed_effects,
        'random_effects': random_effects,
        'aic':            aic,
        'log_likelihood': log_likelihood,
        'fallback':       fallback,
        'response':       response,
        'predictors':     predictors,
        'convergence_ok': convergence_ok,
        }


### 2.2 Convergence Logging

In [ ]:
# Remove existing handlers otherwise warning txt file will be appending all
root_logger = logging.getLogger()
for handler in root_logger.handlers[:]:
    root_logger.removeHandler(handler)
    handler.close()

# Now basicConfig will take effect
logging.basicConfig(
    filename = os.path.join(results_dir, 'RQ1_convergence_log.txt'), 
    level= logging.WARNING,
    format= '%(asctime)s - %(message)s',
    filemode= 'w'
)

warnings.filterwarnings('always', category=ConvergenceWarning)

current_model_context = {}
def custom_warning(msg, *args, **kwargs):
    context = current_model_context
    log_msg = (
        f"response={context.get('response','?')} | "
        f"lag={context.get('lag','?')} | "
        f"pred_set={context.get('pred_set','?')} | "
        f"{str(msg)}"
    )
    logging.warning(log_msg)
    # Short readable print to notebook
    print(f" Convergence issue: {context.get('response','?')} | \
          lag={context.get('lag','?')} | {context.get('pred_set','?')}")

warnings.showwarning = custom_warning


### 2.3 Model Loop

In [ ]:
# Modeling loop

all_fixed = []
all_random = []

for response in RESPONSE_VARS:
    for lag in LAG_WINDOWS:
        for pred_set_name, pred_template_list in PREDICTOR_SETS.items():

            # Resolve actual column names
            predictors = [p.format(lag=lag) for p in pred_template_list]

            # Progress message
            print(f"Running: response={response} | lag={lag} | pred_set={pred_set_name}")

            #Update the model context for logging
            current_model_context = {'response': response, 'lag': lag, 'pred_set': pred_set_name}

            # Fit model
            run_result = fit_mixedlm(df, response, predictors, GROUP_COL)

            # Build fixed effects summary row
            fixed_row = {
                'response':       response,
                'lag':            lag,
                'pred_set':       pred_set_name,
                'fallback':       run_result['fallback'],
                'convergence_ok': run_result['convergence_ok'],
                'aic':            run_result['aic'],
                'log_likelihood': run_result['log_likelihood'],
            }
            for pred, vals in run_result['fixed_effects'].items():
                fixed_row[pred + '_coef']          = vals['coef']
                fixed_row[pred + '_coef_unscaled'] = vals['coef_unscaled']
                fixed_row[pred + '_se']            = vals['se']
                fixed_row[pred + '_pval']          = vals['pval']
                fixed_row[pred + '_ci_low']        = vals['ci_low']
                fixed_row[pred + '_ci_hi']         = vals['ci_hi']
                fixed_row[pred + '_ci_low_unscaled'] = vals['ci_low_unscaled']
                fixed_row[pred + '_ci_hi_unscaled']  = vals['ci_hi_unscaled']
            all_fixed.append(fixed_row)

            # Build random effects rows
            for row in run_result['random_effects']:
                row['response'] = response
                row['lag']      = lag
                row['pred_set'] = pred_set_name
                all_random.append(row)

print(f"\nDone. {len(all_fixed)} model runs completed.")

# Restore default warning handler so it doesn't bleed into later cells
warnings.showwarning = warnings._showwarning_orig

## 3. Results
### Tidying up

In [ ]:
# Random effects df
df_random = pd.DataFrame(all_random)

id_cols = ["eco_id", "response", "lag", "pred_set"]
value_cols = [col for col in df_random.columns if col.endswith("_slope_original")]

df_random_tidy = df_random.melt(id_vars = id_cols,
                               value_vars = value_cols,
                               var_name = "predictors",
                               value_name = "slope").dropna(subset="slope")

df_random_tidy["predictors"] = df_random_tidy["predictors"].str.replace("_slope_original", "")

print(df_random_tidy)


In [ ]:
# Fixed effects df
df_fixed  = pd.DataFrame(all_fixed)

fixed_id_cols = ['response', 'lag', 'pred_set', 'fallback', 'convergence_ok', 'aic', 
                 'log_likelihood']
metrics = ['coef', 'coef_unscaled', 'se', 'pval', 'ci_low', 'ci_hi', 'ci_low_unscaled', 
           'ci_hi_unscaled']

tidy_parts = []
for metric in metrics:
    value_cols_fixed = [col for col in df_fixed.columns if col.endswith(f'_{metric}')]
    melted = df_fixed.melt(
        id_vars=fixed_id_cols,
        value_vars=value_cols_fixed,
        var_name='predictor',
        value_name=metric
    ).dropna(subset=metric)
    melted['predictor'] = melted['predictor'].str.replace(f'_{metric}', '', regex=False)
    tidy_parts.append(melted)

# Merge all metrics together on common columns
merge_cols = fixed_id_cols + ['predictor']
df_fixed_tidy = tidy_parts[0]
for part in tidy_parts[1:]:
    df_fixed_tidy = df_fixed_tidy.merge(part, on=merge_cols, how='outer')

print(df_fixed_tidy)

In [ ]:
# Save em all
df_fixed_tidy.to_csv(os.path.join(results_dir, 'RQ1_fixed_effects_tidy.csv'), index=False)
df_random_tidy.to_csv(os.path.join(results_dir, 'RQ1_random_effects_tidy.csv'), index=False)

print(f'Fixed effects table:  {df_fixed.shape}')
print(f'Random effects table: {df_random.shape}')

### Quick Inspection

In [ ]:
# Check how many runs converged
df_converged = df_fixed_tidy[df_fixed_tidy['convergence_ok'] == True]
print(f'{len(df_converged)} converged runs out of 48')

In [ ]:
# See temp runs
temp_runs = df_fixed_tidy[
    (df_fixed_tidy['pred_set'] == 'temp')]\
        [['response', 'lag', 'predictor', 'coef_unscaled', 'pval', 'convergence_ok']]

print(temp_runs.to_string())

In [ ]:
# See precipitation runs
precip_runs = df_fixed_tidy[
    (df_fixed_tidy['pred_set'] == 'precip')]\
        [['response', 'lag', 'predictor', 'coef_unscaled', 'pval', 'convergence_ok']]

print(precip_runs.to_string())

In [ ]:
# Fixed effect significance summary
sig_df = df_fixed_tidy[df_fixed_tidy['pred_set'] != 'both'].copy()

sig_df['significant']    = sig_df['pval'] < 0.05
sig_df['ci_excl_zero']   = (sig_df['ci_low'] > 0) | (sig_df['ci_hi'] < 0)

summary = sig_df.groupby(['pred_set', 'response', 'lag'])[
    ['significant', 'ci_excl_zero', 'convergence_ok']
].first().reset_index()

print(summary.to_string())

## 4. Visualisation


In [ ]:
# Load results from disk (skip re-running models)
if 'df_fixed_tidy' not in locals():
    df_fixed_tidy = pd.read_csv(os.path.join(results_dir, 'RQ1_fixed_effects_tidy.csv'))
    print(f'Fixed effects loaded from disk: {df_fixed_tidy.shape}')
else:
    print(f'Fixed effects already in environment: {df_fixed_tidy.shape}')

if 'df_random_tidy' not in locals():
    df_random_tidy = pd.read_csv(os.path.join(results_dir, 'RQ1_random_effects_tidy.csv'))
    print(f'Random effects loaded from disk: {df_random_tidy.shape}')
else:
    print(f'Random effects already in environment: {df_random_tidy.shape}')

### 4.1 Heatmap: Fixed Effects


In [ ]:
temp_df   = df_fixed_tidy[(df_fixed_tidy['pred_set'] == 'temp')].copy()
precip_df = df_fixed_tidy[(df_fixed_tidy['pred_set'] == 'precip')].copy()

pivot_temp   = temp_df.pivot(index='response',   columns='lag', values='coef_unscaled')
pivot_precip = precip_df.pivot(index='response', columns='lag', values='coef_unscaled')

mask_temp   = ~temp_df.pivot(index='response',   columns='lag', 
                             values='convergence_ok').astype(bool)
mask_precip = ~precip_df.pivot(index='response', columns='lag', 
                               values='convergence_ok').astype(bool)

def make_annot(pivot, mask):
    annot = pivot.map(lambda v: f'{v:.2f}')
    annot[mask] = annot[mask] + '*'
    return annot

annot_temp   = make_annot(pivot_temp, mask_temp)
annot_precip = make_annot(pivot_precip, mask_precip)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    pivot_temp, annot=annot_temp, fmt='',
    cmap='RdBu', center=0,
    ax=axes[0], cbar=False
)
axes[0].set_title('Temperature slope (days/°C)')
axes[0].set_xlabel('Lag window (days)')
axes[0].set_ylabel('Response variable')

sns.heatmap(
    pivot_precip, annot=annot_precip, fmt='',
    cmap='RdBu', center=0,
    ax=axes[1], cbar=False
)
axes[1].set_title('Precipitation slope (days/mm)')
axes[1].set_xlabel('Lag window (days)')
axes[1].set_ylabel('')

for ax in axes:
    for text in ax.texts:
        if text.get_text().endswith('*'):
            text.set_color('black')

fig.suptitle('Fixed effect slopes by response and lag window\n(* = non-converged)', y=1.02)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'RQ1_01_heatmap_fixed_effects.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Histogram: Slope Distribution

In [ ]:
def plot_histogram(df_random_tidy, response, lag, pred_set, plot_save_dir, all_lags):
    
    # Compute global x range and max count across all lags
    all_slopes = []
    all_counts = []
    for l in all_lags:
        subset = df_random_tidy[
            (df_random_tidy['response'] == response) &
            (df_random_tidy['lag'] == l) &
            (df_random_tidy['pred_set'] == pred_set) &
            (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{l}d')
        ]
        all_slopes.append(subset['slope'])
        counts, _ = np.histogram(subset['slope'], bins=20)
        all_counts.append(counts.max())
    
    all_slopes = pd.concat(all_slopes)
    x_min = all_slopes.min()
    x_max = all_slopes.max()
    global_max = max(all_counts)

    # Filter to current lag
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]
    
    unit = '°C' if pred_set == 'temp' else 'mm'

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(plot_df['slope'], bins=20, edgecolor='black')
    ax.set_xlim(x_min - 0.5, x_max + 0.5)
    ax.set_ylim(0, global_max + 1)
    ax.set_xlabel(f'Per-ecoregion slope (days/{unit})')
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {pred_set} sensitivity of {response} ({lag}d lag)')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, f'RQ1_02_histogram_{response}_{pred_set}_{lag}d.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for pred_set in ["temp", "precip"]:
    for lag in LAG_WINDOWS:
        plot_histogram(df_random_tidy, 'onset_doy', lag, pred_set, plot_dir, LAG_WINDOWS)

### 4.3 Forest Plot

In [ ]:
# Define this ONCE before calling plot_forest in your loop
ALL_BIOMES      = sorted(df['biome_name'].dropna().unique())
PALETTE         = plt.cm.tab10.colors
BIOME_COLOR_MAP = {biome: PALETTE[i % len(PALETTE)] for i, biome in enumerate(ALL_BIOMES)}

def plot_forest(df_random_tidy, df, response, lag, pred_set, plot_save_dir, 
                all_lags, biome_color_map):

    # Compute global x range across all lags
    all_slopes = []
    for l in all_lags:
        subset = df_random_tidy[
            (df_random_tidy['response'] == response) &
            (df_random_tidy['lag'] == l) &
            (df_random_tidy['pred_set'] == pred_set) &
            (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{l}d')
        ]
        all_slopes.append(subset['slope'])

    all_slopes = pd.concat(all_slopes)
    x_margin   = (all_slopes.max() - all_slopes.min()) * 0.05
    x_min      = all_slopes.min() - x_margin
    x_max      = all_slopes.max() + x_margin

    # Filter to current lag
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]

    eco_meta  = df[['eco_id', 'eco_name', 'biome_name']].drop_duplicates()
    forest_df = plot_df[['eco_id', 'slope']].merge(eco_meta, on='eco_id')
    forest_df = forest_df.sort_values('slope')

    colors = [biome_color_map[b] for b in forest_df['biome_name']]

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(range(len(forest_df)), forest_df['slope'], color=colors)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlim(x_min, x_max)
    ax.set_yticks(range(len(forest_df)))
    ax.set_yticklabels(forest_df['eco_name'], fontsize=7)
    ax.set_xlabel('Per-ecoregion slope (days/°C)')
    ax.set_title(f'{response} — {pred_set} sensitivity per ecoregion ({lag}d lag)')

    legend_elements = [Patch(facecolor=biome_color_map[b], label=b) for b in biome_color_map]
    ax.legend(handles=legend_elements, fontsize=7, loc='upper left')

    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, f'RQ1_03_forestplot_{response}_{pred_set}_{lag}d.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for pred_set in ["temp", "precip"]:
    for lag in LAG_WINDOWS:
        plot_forest(df_random_tidy, df, 'onset_doy', lag, pred_set, plot_dir, LAG_WINDOWS, BIOME_COLOR_MAP)

### 4.4 Map

In [ ]:
def plot_map(df_random_tidy, df, gdf, response, lag, pred_set, plot_save_dir, all_lags):
    
    # Compute global slope range across all lags for consistent colormap
    all_slopes = []
    for l in all_lags:
        subset = df_random_tidy[
            (df_random_tidy['response'] == response) &
            (df_random_tidy['lag'] == l) &
            (df_random_tidy['pred_set'] == pred_set) &
            (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{l}d')
        ]
        all_slopes.append(subset['slope'])
    
    all_slopes = pd.concat(all_slopes)
    vmin = all_slopes.min()
    vmax = all_slopes.max()

    # Filter to current lag
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]

    # Merge with metadata and geometries
    eco_meta = df[['eco_id', 'biome_name']].drop_duplicates()
    plot_df = plot_df.merge(eco_meta, on='eco_id')
    map_df = gdf.merge(plot_df[['eco_id', 'slope', 'biome_name']], on='eco_id')

    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    map_df.plot(
        column='slope',
        cmap='RdBu',
        legend=True,
        vmin=vmin,
        vmax=vmax,
        legend_kwds={'label': 'Per-ecoregion slope (days/°C)', 'shrink': 0.6},
        edgecolor='black',
        linewidth=0.3,
        ax=ax
    )
    ax.set_title(f'{pred_set} sensitivity of {response} per ecoregion ({lag}d lag)', fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, f'RQ1_04_map_{response}_{pred_set}_{lag}d.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for pred_set in ["temp", "precip"]:
    for lag in LAG_WINDOWS:
        plot_map(df_random_tidy, df, gdf, 'onset_doy', lag, pred_set, plot_dir, LAG_WINDOWS)

### 4.6 AIC Comparison

In [ ]:
def plot_aic(df_fixed_tidy, plot_save_dir):

    PRED_ORDER  = ['temp', 'precip', 'both']
    PRED_COLORS = {'temp': '#E57373', 'precip': '#64B5F6', 'both': '#81C784'}
    BAR_WIDTH   = 0.25
    OFFSETS     = [-BAR_WIDTH, 0, BAR_WIDTH]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
    axes = axes.flatten()

    # Compute global y max across all response variables for shared axis
    global_max = 0
    for response in RESPONSE_VARS:
        plot_df = (
            df_fixed_tidy[df_fixed_tidy['response'] == response]
            .drop_duplicates(subset=['lag', 'pred_set'])
            [['lag', 'pred_set', 'aic', 'convergence_ok']]
            .copy()
        )
        min_aic = plot_df.groupby('lag')['aic'].min()
        plot_df['delta_aic'] = plot_df.apply(
            lambda row: row['aic'] - min_aic[row['lag']], axis=1
        )
        global_max = max(global_max, plot_df['delta_aic'].max())

    for ax, response in zip(axes, RESPONSE_VARS):

        plot_df = (
            df_fixed_tidy[df_fixed_tidy['response'] == response]
            .drop_duplicates(subset=['lag', 'pred_set'])
            [['lag', 'pred_set', 'aic', 'convergence_ok']]
            .copy()
        )

        lags = sorted(plot_df['lag'].unique())
        x    = np.arange(len(lags))

        min_aic = plot_df.groupby('lag')['aic'].min()
        plot_df['delta_aic'] = plot_df.apply(
            lambda row: row['aic'] - min_aic[row['lag']], axis=1
        )

        for i, pred_set in enumerate(PRED_ORDER):
            for j, lag in enumerate(lags):

                subset = plot_df[
                    (plot_df['pred_set'] == pred_set) & (plot_df['lag'] == lag)
                ]
                if subset.empty:
                    continue

                row       = subset.iloc[0]
                converged = bool(row['convergence_ok'])
                height    = float(row['delta_aic'])
                offset    = x[j] + OFFSETS[i]

                ax.bar(
                    offset,
                    height + 5,
                    bottom=-5,
                    width=BAR_WIDTH * 0.9,
                    color=PRED_COLORS[pred_set],
                    hatch='//' if not converged else None,
                    edgecolor='black' if not converged else 'white',
                    linewidth=0.5
                )

        ax.set_xticks(x)
        ax.set_xticklabels([f'{l}d' for l in lags])
        ax.set_xlabel('Lag window (days)')
        ax.set_ylabel('ΔAIC (from best model at each lag)')
        ax.set_title(response, fontweight='bold')
        ax.set_ylim(bottom=-5, top=global_max * 1.05)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.grid(axis='y', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

    handles = [Patch(facecolor=PRED_COLORS[p], label=p) for p in PRED_ORDER]
    handles += [Patch(facecolor='white', hatch='//', edgecolor='black', label='non-converged')]
    fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=9,
               bbox_to_anchor=(0.5, -0.02))

    fig.suptitle('AIC comparison by predictor set and lag window\n(ΔAIC relative to best model at each lag)',
                 fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, 'RQ1_06_aic_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
plot_aic(df_fixed_tidy, plot_dir)

### 4.7 Biome Boxplot

In [ ]:
def plot_biome_boxplot(df_random_tidy, df, response, lag, pred_set, plot_save_dir, biome_color_map):
    
    plot_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == pred_set) &
        (df_random_tidy['predictors'] == f'{pred_set}_anomaly_{lag}d')
    ]

    eco_meta = df[['eco_id', 'biome_name']].drop_duplicates()
    plot_df  = plot_df.merge(eco_meta, on='eco_id')

    # Sort biomes by median slope
    biome_order = plot_df.groupby('biome_name')['slope'].median().sort_values().index.tolist()

    fig, ax = plt.subplots(figsize=(10, 6))
    
    data_by_biome = [plot_df[plot_df['biome_name'] == b]['slope'].values for b in biome_order]
    bp = ax.boxplot(data_by_biome, vert=False, patch_artist=True, labels=biome_order)

    # Use consistent biome colors from global map
    for patch, biome in zip(bp['boxes'], biome_order):
        patch.set_facecolor(biome_color_map[biome])

    ax.set_xlabel('Per-ecoregion slope (days/°C)' if pred_set == 'temp' else 'Per-ecoregion slope (days/mm)')
    ax.set_title(f'{pred_set} sensitivity of {response} by biome ({lag}d lag)')
    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, f'RQ1_07_biome_boxplot_{response}_{pred_set}_{lag}d.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
plot_biome_boxplot(df_random_tidy, df, "onset_doy", 90, 'temp',   plot_dir, BIOME_COLOR_MAP)
plot_biome_boxplot(df_random_tidy, df, "onset_doy", 90, 'precip', plot_dir, BIOME_COLOR_MAP)

### 4.8 Temp vs Precip Scatter

In [ ]:
def plot_temp_precip_scatter(df_random_tidy, df, response, lag, plot_save_dir, biome_color_map):
    
    temp_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == 'temp') &
        (df_random_tidy['predictors'] == f'temp_anomaly_{lag}d')
    ][['eco_id', 'slope']].rename(columns={'slope': 'temp_slope'})

    precip_df = df_random_tidy[
        (df_random_tidy['response'] == response) &
        (df_random_tidy['lag'] == lag) &
        (df_random_tidy['pred_set'] == 'precip') &
        (df_random_tidy['predictors'] == f'precip_anomaly_{lag}d')
    ][['eco_id', 'slope']].rename(columns={'slope': 'precip_slope'})

    plot_df  = temp_df.merge(precip_df, on='eco_id')
    eco_meta = df[['eco_id', 'biome_name', 'eco_name']].drop_duplicates()
    plot_df  = plot_df.merge(eco_meta, on='eco_id')

    colors = [biome_color_map[b] for b in plot_df['biome_name']]

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(plot_df['temp_slope'], plot_df['precip_slope'],
               c=colors, s=60, edgecolors='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Temperature slope (days/°C)')
    ax.set_ylabel('Precipitation slope (days/mm)')
    ax.set_title(f'Temp vs precip sensitivity per ecoregion — {response} ({lag}d lag)')

    legend_elements = [Patch(facecolor=biome_color_map[b], label=b) for b in biome_color_map]
    ax.legend(handles=legend_elements, fontsize=7)

    plt.tight_layout()
    plt.savefig(os.path.join(plot_save_dir, f'RQ1_08_temp_precip_scatter_{response}_{lag}d.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
plot_temp_precip_scatter(df_random_tidy, df, 'onset_doy', 90, plot_dir, BIOME_COLOR_MAP)

## 5: ROBUSTNESS CHECKS

In [ ]:
# Primary specification
# All robustness checks below are run against this single spec.
# Change these values to re-run checks for a different response/lag/predictor.

PRIMARY_RESPONSE  = 'onset_doy'
PRIMARY_LAG       = 90
PRIMARY_PRED_SET  = 'temp'
PRIMARY_PREDICTOR = f'temp_anomaly_{PRIMARY_LAG}d'

In [ ]:
# Filter df_random_tidy to the primary spec — one row per ecoregion
rob_df = df_random_tidy[
    (df_random_tidy['response']   == PRIMARY_RESPONSE) &
    (df_random_tidy['lag']        == PRIMARY_LAG) &
    (df_random_tidy['pred_set']   == PRIMARY_PRED_SET) &
    (df_random_tidy['predictors'] == PRIMARY_PREDICTOR)
].copy()

# Flag ecoregions whose slope is more than 2 SD from the mean.
# 2 SD captures ~95% of a normal distribution — anything outside is
# considered extreme enough to warrant scrutiny.
slope_mean = rob_df['slope'].mean()
slope_std  = rob_df['slope'].std()
rob_df['outlier_flag'] = (rob_df['slope'] - slope_mean).abs() > 2 * slope_std

print(f"Slope mean: {slope_mean:.2f} days/°C")
print(f"Slope std:  {slope_std:.2f} days/°C")
print(f"Outliers flagged: {rob_df['outlier_flag'].sum()} / {len(rob_df)} ecoregions")
print()
print(rob_df[rob_df['outlier_flag']][['eco_id', 'slope']])

In [ ]:
# For each ecoregion, compute Spearman ρ between temperature anomaly and onset DOY independently, 
# completely ignoring the mixed model. This is a model-free sanity check: if β and ρ agree in sign,
# the mixed model result is not an artefact.

spearman_rows = []

for eco_id, group in df.groupby('eco_id'):
    x = group[PRIMARY_PREDICTOR].dropna()
    y = group[PRIMARY_RESPONSE].dropna()

    common_idx = x.index.intersection(y.index)
    x = x.loc[common_idx]
    y = y.loc[common_idx]

    if len(x) < 5:
        continue

    rho, pval = spearmanr(x, y)
    spearman_rows.append({'eco_id': eco_id, 'spearman_rho': rho, 'spearman_pval': pval})

spearman_df = pd.DataFrame(spearman_rows)
print(f"Spearman correlations computed for {len(spearman_df)} ecoregions.")

In [ ]:
# Shared ecoregion metadata — used throughout the robustness section
eco_meta = df[['eco_id', 'eco_name', 'biome_name']].drop_duplicates()

# Merge mixed model slopes and Spearman into one table
df_robust = (
    rob_df[['eco_id', 'slope', 'outlier_flag']]
    .merge(spearman_df, on='eco_id', how='inner')
    .merge(eco_meta,    on='eco_id', how='left')
)

# Sign agreement: do β and ρ point in the same direction?
df_robust['sign_agreement'] = (
    np.sign(df_robust['slope']) == np.sign(df_robust['spearman_rho'])
)

# Separate failure reasons explicitly
df_robust['fails_outlier']          = df_robust['outlier_flag']
df_robust['fails_sign_agreement']   = ~df_robust['sign_agreement']

# passes_all: must pass both checks
df_robust['passes_all'] = (
    ~df_robust['outlier_flag'] &
     df_robust['sign_agreement']
)

# Summary printout
print(df_robust[['eco_name', 'slope', 'spearman_rho',
                 'outlier_flag', 'sign_agreement', 'passes_all']].sort_values('slope').to_string())
print()
print(f"Ecoregions passing robustness checks: {df_robust['passes_all'].sum()} / {len(df_robust)}")
print(f"  - Fails outlier check only:         {(df_robust['fails_outlier'] & df_robust['sign_agreement']).sum()}")
print(f"  - Fails sign agreement only:        {(~df_robust['fails_outlier'] & df_robust['fails_sign_agreement']).sum()}")
print(f"  - Fails both:                       {(df_robust['fails_outlier'] & df_robust['fails_sign_agreement']).sum()}")


df_robust.to_csv(os.path.join(results_dir, 'rq1_robustness_summary.csv'), index=False)
print("\nSaved: rq1_robustness_summary.csv")

In [ ]:
# Group 1: Outlier flag
# These ecoregions have extreme slopes but sign agreement is intact.
# The mixed model may be overreaching in data-sparse ecoregions.
outlier_failures = df_robust[df_robust['fails_outlier']]

print(f"Group 1 — Outlier flag ({len(outlier_failures)} ecoregions):")
print("Both methods agree on direction but slope is numerically extreme.\n")
print(outlier_failures[['eco_name', 'biome_name', 'slope', 'spearman_rho']].sort_values('slope').to_string())

In [ ]:
# Group 2: Sign disagreement 
# These ecoregions have β and ρ pointing in opposite directions.
# Likely candidates: desert/marginal ecoregions where fire timing
# is unreliable or temperature is not the primary driver.
sign_failures = df_robust[df_robust['fails_sign_agreement'] & ~df_robust['fails_outlier']]

print(f"Group 2 — Sign disagreement ({len(sign_failures)} ecoregions):")
print("Mixed model β and Spearman ρ point in opposite directions.\n")
print(sign_failures[['eco_name', 'biome_name', 'slope', 'spearman_rho']].sort_values('slope').to_string())

In [ ]:
# Biome breakdown of sign disagreements
# Are the sign disagreements concentrated in specific biomes?
print("Biome breakdown of sign disagreement failures:")
print()
print(sign_failures['biome_name'].value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for biome, group in df_robust.groupby('biome_name'):
    ax.scatter(group['spearman_rho'], group['slope'],
               color=BIOME_COLOR_MAP[biome], label=biome,
               s=50, edgecolors='black', linewidth=0.5, alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Spearman ρ  (per-ecoregion, independent)')
ax.set_ylabel('Mixed model β  (days/°C)')
ax.set_title(f'Robustness Check: Mixed Model β vs Spearman ρ\n{PRIMARY_RESPONSE} ~ {PRIMARY_PREDICTOR}')
ax.legend(fontsize=7, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'RQ1_robustness_spearman_vs_beta.png'), dpi=150, bbox_inches='tight')
plt.show()

# Overall agreement statistic
rho_vs_beta, p = spearmanr(df_robust['slope'], df_robust['spearman_rho'])
print(f"Agreement between β and ρ across ecoregions: ρ = {rho_vs_beta:.3f}, p = {p:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

gdf_rob = gdf.merge(df_robust[['eco_id', 'passes_all']], on='eco_id', how='left')
gdf_rob['passes_all'] = gdf_rob['passes_all'].fillna(False)

gdf_rob[~gdf_rob['passes_all']].plot(ax=ax, color='#BDBDBD', linewidth=0.3)
gdf_rob[ gdf_rob['passes_all']].plot(ax=ax, color='#2196F3', linewidth=0.3)

legend_elements = [
    Patch(facecolor='#2196F3', label=f'Passes all checks (n={df_robust["passes_all"].sum()})'),
    Patch(facecolor='#BDBDBD', label=f'Fails ≥1 check   (n={(~df_robust["passes_all"]).sum()})'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower left')
ax.set_title(f'Spatial Distribution of Robust Ecoregions\n{PRIMARY_RESPONSE} ~ {PRIMARY_PREDICTOR}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'rq1_robustness_map.png'), dpi=150, bbox_inches='tight')
plt.show()